# Governance — one contract, two engines

The **same** `sales_grants.json` that governs Unity Catalog governs Snowflake.
Here, live: Snowflake reads the gold layer Databricks wrote — **zero copy** —
and the same PII masking is enforced per role. Hit **Run all**.


## 1 · Zero-copy read of the Databricks gold layer
Setup runs as `ACCOUNTADMIN`; the read runs as `DEV_ANALYSTS`.


In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE DEV_SALES_WH;
USE DATABASE "sales_aws";
CREATE SCHEMA IF NOT EXISTS "sales_aws"."demo";
CREATE FILE FORMAT IF NOT EXISTS "sales_aws"."demo"."parquet_ff" TYPE = PARQUET;
CREATE OR REPLACE EXTERNAL TABLE "sales_aws"."demo"."executive_cross_cloud" (
    market STRING AS (VALUE:market::STRING),
    revenue DOUBLE AS (VALUE:revenue::DOUBLE),
    marketing_roi DOUBLE AS (VALUE:marketing_roi::DOUBLE),
    stockout_risk STRING AS (VALUE:stockout_risk::STRING),
    revenue_at_risk DOUBLE AS (VALUE:revenue_at_risk::DOUBLE) )
  LOCATION = @"sales_aws"."_EXTERNAL"."loc_sales_gold"/executive/
  FILE_FORMAT = "sales_aws"."demo"."parquet_ff" AUTO_REFRESH = FALSE PATTERN = '.*[.]parquet';
GRANT USAGE ON SCHEMA "sales_aws"."demo" TO ROLE DEV_ANALYSTS;
GRANT SELECT ON EXTERNAL TABLE "sales_aws"."demo"."executive_cross_cloud" TO ROLE DEV_ANALYSTS;


**Read it — three clouds fused by market, queried by a fourth engine:**


In [ ]:
USE ROLE DEV_ANALYSTS;
SELECT market, revenue, marketing_roi, stockout_risk, revenue_at_risk
FROM "sales_aws"."demo"."executive_cross_cloud" ORDER BY revenue DESC;


**Where does it live? Not in Snowflake — in S3, where Databricks wrote it:**


In [ ]:
SELECT DISTINCT metadata$filename AS s3_object
FROM "sales_aws"."demo"."executive_cross_cloud";


## 2 · Column-level masking
The governed catalogs hold **no** PII (it never leaves Postgres). This demo
schema exists only to show the mechanism. Same query, different role.


In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE TABLE "sales_aws"."demo"."customers" AS
SELECT 'cust_' || seq4() AS customer_id,
       'customer' || uniform(1,99999,random()) || '@example.com' AS email,
       ARRAY_CONSTRUCT('Germany','France','Netherlands','Spain','Italy','Poland')[uniform(0,5,random())]::STRING AS market
FROM TABLE(generator(rowcount => 200));
CREATE OR REPLACE MASKING POLICY "sales_aws"."demo"."pii_email_mask"
  AS (val STRING) RETURNS STRING ->
  CASE WHEN CURRENT_ROLE() = 'DEV_METASTORE_ADMINS' THEN val ELSE '***MASKED***' END;
ALTER TABLE "sales_aws"."demo"."customers" MODIFY COLUMN email SET MASKING POLICY "sales_aws"."demo"."pii_email_mask";
GRANT USAGE ON SCHEMA "sales_aws"."demo" TO ROLE DEV_ANALYSTS;
GRANT SELECT ON TABLE "sales_aws"."demo"."customers" TO ROLE DEV_ANALYSTS;
GRANT USAGE ON SCHEMA "sales_aws"."demo" TO ROLE DEV_METASTORE_ADMINS;
GRANT SELECT ON TABLE "sales_aws"."demo"."customers" TO ROLE DEV_METASTORE_ADMINS;


**As ADMIN — sees the real email:**


In [ ]:
USE ROLE DEV_METASTORE_ADMINS;
SELECT customer_id, market, email FROM "sales_aws"."demo"."customers" LIMIT 5;


**As ANALYST — same table, PII masked:**


In [ ]:
USE ROLE DEV_ANALYSTS;
SELECT customer_id, market, email FROM "sales_aws"."demo"."customers" LIMIT 5;


**The analyst still does their job — aggregate by market, no identities:**


In [ ]:
SELECT market, count(*) AS customers
FROM "sales_aws"."demo"."customers" GROUP BY market ORDER BY customers DESC;


---
**One contract, two engines.** `python3 scripts/snowflake_backend.py --check`
proves the Unity Catalog and Snowflake grants are access-equivalent — and
reports the one distinction Snowflake cannot express, instead of hiding it.
